1. Изучить [теорию по анализу](https://colab.research.google.com/drive/1tzekUR7XYY6UixcwWQwIjs4wi_FQBoIe#scrollTo=RjJocNVNQFyw&uniqifier=1)
2. [Теорию по очистке данных](https://colab.research.google.com/drive/1tgmKQomZkLssQvwSMsk0weG9MvqEekwM?usp=sharing)

# Исследование объявлений о продаже квартир

## Общее описание проекта
В распоряжении данные сервиса о продаже квартир в Санкт-Петербурге и соседних населённых пунктов за несколько лет. Нужно изучить влияние различных факторов на рыночную стоимость объектов недвижимости.

## Описание признакового пространства
По каждой квартире на продажу доступны следующие признаки.

|Признак|Описание признака|
|-------------:|:------------|
|airports_nearest|расстояние до ближайшего аэропорта в метрах (м)|
|balcony|число балконов|
|ceiling_height|высота потолков (м)|
|cityCenters_nearest|расстояние до центра города (м)|
|days_exposition| сколько дней было размещено объявление (от публикации до снятия)|
|first_day_exposition|дата публикации|
|floor|этаж|
|floors_total| всего этажей в доме|
|is_apartment|апартаменты (булев тип)|
|kitchen_area|площадь кухни в квадратных метрах (м²)|
|last_price|цена на момент снятия с публикации|
|living_area|жилая площадь в квадратных метрах(м²)|
|locality_name|название населённого пункта|
|open_plan|свободная планировка (булев тип)|
|parks_around3000|число парков в радиусе 3 км|
|parks_nearest|расстояние до ближайшего парка (м)|
|ponds_around3000|число водоёмов в радиусе 3 км|
|ponds_nearest|расстояние до ближайшего водоёма (м)|
|rooms|число комнат|
|studio|квартира-студия (булев тип)|
|total_area|площадь квартиры в квадратных метрах (м²)|
|total_images|число фотографий квартиры в объявлении|

Пояснение: апартаменты — это нежилые помещения, не относящиеся к жилому фонду, но имеющие необходимые условия для проживания.

## Инструкции по выполнению проекта

Шаг 1. Откройте файл с данными и изучите общую информацию

Шаг 2. Предобработка данных
- Обработать аномальные наблюдения;
- Привести данные к необходимым типам;
- Исследовать дублирующиеся записи;
- Обработать пропущенные значения.

Шаг 3. Посчитайте и добавьте в таблицу:
цену квадратного метра;
день недели, месяц и год публикации объявления;
этаж квартиры; варианты — первый, последний, другой;
соотношение жилой и общей площади, а также отношение площади кухни к общей.

Шаг 4. Проведите исследовательский анализ данных и выполните инструкции:
- Изучите следующие параметры: площадь, цена, число комнат, высота потолков. Постройте гистограммы для каждого параметра.
- Изучите время продажи квартиры. Постройте гистограмму. Посчитайте среднее и медиану.
- Какие факторы больше всего влияют на стоимость квартиры? Изучите, зависит ли цена от площади, числа комнат, удалённости от центра. Изучите зависимость цены от того, на каком этаже расположена квартира: первом, последнем или другом. Также изучите зависимость от даты размещения: дня недели, месяца и года.
- Выберите 10 населённых пунктов с наибольшим числом объявлений. Посчитайте среднюю цену квадратного метра в этих населённых пунктах. Выделите среди них населённые пункты с самой высокой и низкой стоимостью жилья. Эти данные можно найти по имени в столбце 'locality_name'.
- Изучите предложения квартир: для каждой квартиры есть информация о расстоянии до центра. Выделите квартиры в Санкт-Петербурге ('locality_name').
- Выделите сегмент квартир в центре. Проанализируйте эту территорию и изучите следующие параметры: площадь, цена, число комнат, высота потолков. Также выделите факторы, которые влияют на стоимость квартиры (число комнат, этаж, удалённость от центра, дата размещения объявления).


# Шаг 1. Откройте файл с данными и изучите общую информацию.
<a class="anchor" id="step1"></a>

## Импорт библиотек

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt


In [ ]:
CAT_OS = 'https://github.com/OlesiaAngel/DataAnalitics/blob/main/%D0%B4%D0%B0%D0%BD%D0%BD%D1%8B%D0%B5/'


In [ ]:
data = pd.read_csv(CAT_OS+'predobr/p7.csv?raw=True', sep='\t')


In [ ]:
# Общая информация о датасете
print('Форма датасета:', data.shape)
print('\nПервые строки:')
display(data.head())
print('\nИнформация о столбцах:')
data.info()
print('\nСтатистическое описание:')
display(data.describe())


# Шаг 2. Предобработка данных

In [ ]:
# 2.1 Дубликаты
print('Количество дубликатов:', data.duplicated().sum())
data = data.drop_duplicates()
print('После удаления дубликатов:', data.shape)


In [ ]:
# 2.2 Пропущенные значения
missing = data.isnull().sum()
missing_pct = (missing / len(data) * 100).round(2)
missing_df = pd.DataFrame({'Пропущено': missing, 'Процент (%)': missing_pct})
missing_df = missing_df[missing_df['Пропущено'] > 0].sort_values('Процент (%)', ascending=False)
print('Пропущенные значения:')
display(missing_df)


In [ ]:
# 2.3 Обработка аномальных значений
# Высота потолков: допустимый диапазон 2.0 — 6.0 м
print('Статистика ceiling_height до очистки:')
print(data['ceiling_height'].describe())
data.loc[data['ceiling_height'] > 6.0, 'ceiling_height'] = np.nan
data.loc[data['ceiling_height'] < 2.0, 'ceiling_height'] = np.nan
print('\nСтатистика ceiling_height после очистки:')
print(data['ceiling_height'].describe())

# Число комнат: не более 20
data.loc[data['rooms'] > 20, 'rooms'] = np.nan


In [ ]:
# 2.4 Приведение типов
# Дата публикации -> datetime
data['first_day_exposition'] = pd.to_datetime(data['first_day_exposition'], errors='coerce')

# Заполнение пропущенных значений
# Медиана для числовых признаков
num_cols_fill = ['balcony', 'ceiling_height', 'floors_total', 'living_area',
                  'airports_nearest', 'cityCenters_nearest',
                  'parks_nearest', 'ponds_nearest', 'days_exposition']
for col in num_cols_fill:
    if col in data.columns:
        median_val = data[col].median()
        data[col] = data[col].fillna(median_val)
        print(f'{col}: заполнено медианой = {median_val:.2f}')

# Булевы признаки: False
for col in ['is_apartment', 'open_plan', 'studio']:
    if col in data.columns:
        data[col] = data[col].fillna(False)

print('\nПосле обработки пропущенных значений:')
print(data.isnull().sum())


# Шаг 3. Добавление новых признаков

In [ ]:
# 3.1 Цена квадратного метра
data['price_per_sqm'] = data['last_price'] / data['total_area']

# 3.2 День недели, месяц и год публикации объявления
data['pub_weekday'] = data['first_day_exposition'].dt.dayofweek
data['pub_month'] = data['first_day_exposition'].dt.month
data['pub_year'] = data['first_day_exposition'].dt.year

# 3.3 Категория этажа
def floor_category(row):
    if row['floor'] == 1:
        return 'первый'
    elif row['floor'] == row['floors_total']:
        return 'последний'
    else:
        return 'другой'

data['floor_type'] = data.apply(floor_category, axis=1)

# 3.4 Отношения площадей
data['living_to_total_ratio'] = data['living_area'] / data['total_area']
data['kitchen_to_total_ratio'] = data['kitchen_area'] / data['total_area']

print('Новые признаки добавлены:')
display(data[['price_per_sqm', 'pub_weekday', 'pub_month', 'pub_year',
              'floor_type', 'living_to_total_ratio', 'kitchen_to_total_ratio']].head())


# Шаг 4. Исследовательский анализ данных

In [ ]:
# 4.1 Гистограммы: площадь, цена, число комнат, высота потолков
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

params = [
    ('total_area', 'Площадь квартиры (м²)', 0, 300),
    ('last_price', 'Цена (руб.)', 0, data['last_price'].quantile(0.99)),
    ('rooms', 'Число комнат', None, None),
    ('ceiling_height', 'Высота потолков (м)', None, None),
]

for ax, (col, title, xlim_min, xlim_max) in zip(axes.flatten(), params):
    ax.hist(data[col].dropna(), bins=50, color='steelblue', edgecolor='white', alpha=0.8)
    ax.set_title(title, fontsize=12)
    ax.set_xlabel(col)
    ax.set_ylabel('Количество объявлений')
    if xlim_min is not None:
        ax.set_xlim(xlim_min, xlim_max)
    ax.axvline(data[col].median(), color='red', linestyle='--', linewidth=1.5, label=f'Медиана: {data[col].median():.0f}')
    ax.legend(fontsize=9)

plt.suptitle('Распределение основных параметров квартир', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


In [ ]:
# 4.2 Время продажи квартиры (days_exposition)
fig, ax = plt.subplots(figsize=(12, 5))
days = data['days_exposition'].dropna()
ax.hist(days[days <= 365*2], bins=60, color='coral', edgecolor='white', alpha=0.8)
ax.set_title('Распределение времени продажи квартиры', fontsize=13)
ax.set_xlabel('Дней на продажу')
ax.set_ylabel('Количество объявлений')
mean_days = days.mean()
median_days = days.median()
ax.axvline(mean_days, color='blue', linestyle='--', linewidth=2, label=f'Среднее: {mean_days:.0f} дней')
ax.axvline(median_days, color='green', linestyle='--', linewidth=2, label=f'Медиана: {median_days:.0f} дней')
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()
print(f'Среднее время продажи: {mean_days:.1f} дней')
print(f'Медианное время продажи: {median_days:.1f} дней')


In [ ]:
# 4.3 Факторы, влияющие на стоимость квартиры
# Зависимость от площади, числа комнат и удалённости от центра
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

data_clean = data[data['last_price'] <= data['last_price'].quantile(0.99)]

# Площадь vs Цена
axes[0].scatter(data_clean['total_area'], data_clean['last_price'],
                alpha=0.3, s=10, color='steelblue')
axes[0].set_xlabel('Площадь (м²)')
axes[0].set_ylabel('Цена (руб.)')
axes[0].set_title('Площадь vs Цена')

# Число комнат vs Медианная цена
rooms_price = data_clean.groupby('rooms')['last_price'].median().reset_index()
axes[1].bar(rooms_price['rooms'].astype(str), rooms_price['last_price'],
            color='coral', alpha=0.8)
axes[1].set_xlabel('Число комнат')
axes[1].set_ylabel('Медианная цена (руб.)')
axes[1].set_title('Число комнат vs Медианная цена')

# Удалённость от центра vs Цена
axes[2].scatter(data_clean['cityCenters_nearest'] / 1000, data_clean['last_price'],
                alpha=0.3, s=10, color='green')
axes[2].set_xlabel('Расстояние до центра (км)')
axes[2].set_ylabel('Цена (руб.)')
axes[2].set_title('Удалённость от центра vs Цена')

plt.suptitle('Факторы, влияющие на стоимость квартиры', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


In [ ]:
# Зависимость цены от типа этажа
fig, ax = plt.subplots(figsize=(9, 5))
floor_order = ['первый', 'другой', 'последний']
floor_labels = {'первый': 'Первый этаж', 'другой': 'Другой этаж', 'последний': 'Последний этаж'}
floor_price = data_clean.groupby('floor_type')['last_price'].median().reindex(floor_order)
ax.bar([floor_labels[k] for k in floor_order], floor_price.values,
       color=['#e74c3c', '#3498db', '#2ecc71'], alpha=0.85)
ax.set_title('Медианная цена по типу этажа', fontsize=12)
ax.set_ylabel('Медианная цена (руб.)')
for i, v in enumerate(floor_price.values):
    ax.text(i, v + 50000, f'{v/1e6:.2f} млн', ha='center', fontsize=11)
plt.tight_layout()
plt.show()

# Зависимость от даты размещения
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
weekday_names = ['Пн', 'Вт', 'Ср', 'Чт', 'Пт', 'Сб', 'Вс']

wd_price = data_clean.groupby('pub_weekday')['last_price'].median()
axes[0].bar([weekday_names[i] for i in wd_price.index], wd_price.values, color='steelblue', alpha=0.8)
axes[0].set_title('Медианная цена по дню недели')
axes[0].set_ylabel('Цена (руб.)')

m_price = data_clean.groupby('pub_month')['last_price'].median()
axes[1].bar(m_price.index.astype(str), m_price.values, color='coral', alpha=0.8)
axes[1].set_title('Медианная цена по месяцу')
axes[1].set_xlabel('Месяц')

y_price = data_clean.groupby('pub_year')['last_price'].median()
axes[2].bar(y_price.index.astype(str), y_price.values, color='green', alpha=0.8)
axes[2].set_title('Медианная цена по году')
axes[2].set_xlabel('Год')

plt.suptitle('Зависимость цены от даты публикации', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


In [ ]:
# 4.4 Топ-10 населённых пунктов по числу объявлений
top10_localities = data['locality_name'].value_counts().head(10).index
top10_data = data[data['locality_name'].isin(top10_localities)]

locality_stats = top10_data.groupby('locality_name').agg(
    Объявлений=('locality_name', 'count'),
    Средняя_цена_кв_м=('price_per_sqm', 'mean')
).sort_values('Объявлений', ascending=False)

print('Топ-10 населённых пунктов:')
display(locality_stats)

print(f"\nСамый дорогой: {locality_stats['Средняя_цена_кв_м'].idxmax()} — "
      f"{locality_stats['Средняя_цена_кв_м'].max():,.0f} руб/м²")
print(f"Самый дешёвый: {locality_stats['Средняя_цена_кв_м'].idxmin()} — "
      f"{locality_stats['Средняя_цена_кв_м'].min():,.0f} руб/м²")

fig, ax = plt.subplots(figsize=(12, 6))
locality_stats_sorted = locality_stats.sort_values('Средняя_цена_кв_м', ascending=True)
ax.barh(locality_stats_sorted.index, locality_stats_sorted['Средняя_цена_кв_м'],
        color='steelblue', alpha=0.8)
ax.set_title('Средняя цена м² по населённым пунктам (топ-10)', fontsize=12)
ax.set_xlabel('Средняя цена (руб/м²)')
plt.tight_layout()
plt.show()


In [ ]:
# 4.5 Анализ квартир в Санкт-Петербурге
spb_data = data[data['locality_name'] == 'Санкт-Петербург'].copy()
print(f'Квартиры в Санкт-Петербурге: {len(spb_data)} объявлений')
display(spb_data[['total_area', 'last_price', 'rooms', 'ceiling_height',
                   'cityCenters_nearest', 'floor_type']].describe())

# Порог центра: квартиры, расположенные в пределах 3 км от центра
CENTER_THRESHOLD = 3000
spb_center = spb_data[spb_data['cityCenters_nearest'] <= CENTER_THRESHOLD].copy()
print(f'\nКвартиры в центре СПб (до {CENTER_THRESHOLD/1000:.0f} км): {len(spb_center)} объявлений')

# Параметры центральных квартир
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

params_center = [
    ('total_area', 'Площадь (м²)'),
    ('last_price', 'Цена (руб.)'),
    ('rooms', 'Число комнат'),
    ('ceiling_height', 'Высота потолков (м)'),
]

for ax, (col, title) in zip(axes.flatten(), params_center):
    ax.hist(spb_center[col].dropna(), bins=40, color='purple', edgecolor='white', alpha=0.7)
    ax.set_title(f'Центр СПб: {title}', fontsize=11)
    ax.set_xlabel(col)
    ax.set_ylabel('Количество')
    ax.axvline(spb_center[col].median(), color='red', linestyle='--',
               label=f'Медиана: {spb_center[col].median():.1f}')
    ax.legend(fontsize=9)

plt.suptitle('Квартиры в центре Санкт-Петербурга', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


In [ ]:
# Факторы, влияющие на стоимость квартир в центре СПб
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
spb_center_clean = spb_center[spb_center['last_price'] <= spb_center['last_price'].quantile(0.99)]

# Число комнат vs Цена
rooms_price_c = spb_center_clean.groupby('rooms')['last_price'].median()
axes[0, 0].bar(rooms_price_c.index.astype(str), rooms_price_c.values, color='coral', alpha=0.8)
axes[0, 0].set_title('Число комнат vs Цена (центр)')
axes[0, 0].set_xlabel('Число комнат')
axes[0, 0].set_ylabel('Медианная цена (руб.)')

# Тип этажа vs Цена
ft_price_c = spb_center_clean.groupby('floor_type')['last_price'].median().reindex(floor_order)
axes[0, 1].bar([floor_labels[k] for k in floor_order], ft_price_c.values,
               color=['#e74c3c', '#3498db', '#2ecc71'], alpha=0.8)
axes[0, 1].set_title('Тип этажа vs Цена (центр)')
axes[0, 1].set_ylabel('Медианная цена (руб.)')

# Удалённость от центра vs Цена
axes[1, 0].scatter(spb_center_clean['cityCenters_nearest'] / 1000,
                   spb_center_clean['last_price'],
                   alpha=0.4, s=15, color='steelblue')
axes[1, 0].set_xlabel('Расстояние до центра (км)')
axes[1, 0].set_ylabel('Цена (руб.)')
axes[1, 0].set_title('Удалённость от центра vs Цена')

# Год публикации vs Медианная цена в центре
yr_price_c = spb_center_clean.groupby('pub_year')['last_price'].median()
axes[1, 1].plot(yr_price_c.index, yr_price_c.values, marker='o', color='green', linewidth=2)
axes[1, 1].set_xlabel('Год')
axes[1, 1].set_ylabel('Медианная цена (руб.)')
axes[1, 1].set_title('Год публикации vs Цена (центр)')

plt.suptitle('Факторы стоимости квартир в центре СПб', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()
print('Анализ завершён!')
